# AI Red Teaming Agent for Generative AI models and applications in Azure AI Foundry

## Objective
This notebook walks through how to use Azure AI Evaluation's AI Red Teaming Agent functionality to assess the safety and resilience of AI systems against adversarial prompt attacks. AI Red Teaming Agent leverages [Risk and Safety Evaluations](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-metrics-built-in?tabs=warning#risk-and-safety-evaluators) to help identify potential safety issues across different risk categories (violence, hate/unfairness, sexual content, self-harm) combined with attack strategies of varying complexity levels from [PyRIT](https://github.com/Azure/PyRIT), Microsoft AI Red Teaming team's open framework for automated AI red teaming.

## Time
You should expect to spend about 30-45 minutes running this notebook. Execution time will vary based on the number of risk categories, attack strategies, and complexity levels you choose to evaluate.

## Before you begin

### Prerequisite
First, if you have an Azure subscription, create an [Azure AI hub](https://learn.microsoft.com/en-us/azure/ai-studio/concepts/ai-resources) then [create an Azure AI project](https://learn.microsoft.com/en-us/azure/ai-studio/concepts/ai-resources). AI projects and Hubs can be served within a private network and are compatible with private endpoints. You **do not** need to provide your own LLM deployment as the AI Red Teaming Agent hosts adversarial models for both simulation and evaluation of harmful content and connects to it via your Azure AI project.

In order to upload your results to Azure AI Foundry:
- Your AI Foundry project must have a connection (*Connected Resources*) to a storage account with `Microsoft Entra ID` authentication enabled.
- Your AI Foundry project must have the `Storage Blob Data Contributor` role in the storage account.
- You must have the `Storage Blob Data Contributor` role in the storage account.
- You must have network access to the storage account.

For more information see: https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/run-scans-ai-red-teaming-agent

**Important**: First, ensure that you've installed the [Azure CLI](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli) and then make sure to authenticate to Azure using `az login` in your terminal before running this notebook.

### Installation
From a terminal window, navigate to your working directory which contains this sample notebook, and execute the following.
```bash
python -m venv .venv
```

Then, activate the virtual environment created:

```bash
# %source .venv/bin/activate # If using Mac/Linux OS
.venv/Scripts/activate # If using Windows OS
```

With your virtual environment activated, install the following packages required to execute this notebook:

```bash
pip install uv
uv pip install 'azure-ai-evaluation[redteam]' azure-identity openai azure-ai-projects
```


Now open VSCode with the following command, and ensure your virtual environment is used as kernel to run the remainder of this notebook.
```bash
code .
```

### Imports

In [2]:
from typing import Optional, Dict, Any
import os

# Azure imports
from azure.ai.evaluation.red_team import RedTeam, RiskCategory, AttackStrategy

# OpenAI imports
from openai import AzureOpenAI

### Login to Azure with valid credentials

Ensure that you've installed the [Azure CLI](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli) and then make sure to authenticate to Azure using `az login` in your terminal before running this notebook.

Configure the `credential` object with a different AzureCredential type if this is a requirement for your environment.

In [3]:
# Import required packages
from IPython.display import display
from IPython.display import HTML
import getpass
from dotenv import load_dotenv
import os
from pathlib import Path  # For cross-platform path handling

# Get the path to the .env file which is in the parent directory
notebook_path = Path().absolute()  # Get absolute path of current notebook
parent_dir = notebook_path.parent  # Get parent directory
load_dotenv(parent_dir / '.env')  # Load environment variables from .env file

# Get tenant ID from environment variable
tenant_id = os.getenv("TENANT_ID")

# Azure login with specific tenant
!az login --tenant {tenant_id}

# Get subscription ID from environment variable (it's already configured separately)
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
conn_str = os.getenv("PROJECT_CONNECTION_STRING")

print(f"📋 Connection String: {conn_str}")
print(f"📋 Subscription ID: {subscription_id}")

if subscription_id:
    # Set the subscription
    !az account set --subscription {subscription_id}
    print(f"✓ Successfully set subscription: {subscription_id}")
else:
    print("⚠️ AZURE_SUBSCRIPTION_ID not found in environment variables")
    print("   Check your .env file to make sure AZURE_SUBSCRIPTION_ID is set")

A web browser has been opened at https://login.microsoftonline.com/16b3c013-d300-468d-ac64-7eda0820b6d3/oauth2/v2.0/authorize. Please continue the login in the web browser. If no web browser is available or if the web browser fails to open, use device code flow with `az login --use-device-code`.
/usr/bin/xdg-open: 882: x-www-browser: not found
/usr/bin/xdg-open: 882: firefox: not found
/usr/bin/xdg-open: 882: iceweasel: not found
/usr/bin/xdg-open: 882: seamonkey: not found
/usr/bin/xdg-open: 882: mozilla: not found
/usr/bin/xdg-open: 882: epiphany: not found
/usr/bin/xdg-open: 882: konqueror: not found
/usr/bin/xdg-open: 882: chromium: not found
/usr/bin/xdg-open: 882: chromium-browser: not found
/usr/bin/xdg-open: 882: google-chrome: not found

Retrieving subscriptions for the selection...

[Tenant and subscription selection]

No     Subscription name      Subscription ID                       Tenant
-----  ---------------------  ------------------------------------  ----------------

In [4]:
# Then use DefaultAzureCredential in your code
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AccessToken
import logging

# Enable detailed logging
logging.basicConfig(level=logging.DEBUG)

try:
    credential = DefaultAzureCredential()
    # Test token acquisition
    token = credential.get_token("https://cognitiveservices.azure.com/.default")
    print("Successfully acquired token!")
except Exception as e:
    print(f"Authentication failed: {str(e)}")

INFO:azure.identity._credentials.environment:No environment configuration found.
INFO:azure.identity._credentials.managed_identity:ManagedIdentityCredential will use IMDS
DEBUG:azure.identity._internal.decorators:EnvironmentCredential.get_token failed: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
Traceback (most recent call last):
  File "/home/brian/working/ai-foundry-e2e-lab/.venv/lib/python3.10/site-packages/azure/identity/_internal/decorators.py", line 23, in wrapper
    token = fn(*args, **kwargs)
  File "/home/brian/working/ai-foundry-e2e-lab/.venv/lib/python3.10/site-packages/azure/identity/_credentials/environment.py", line 155, in get_token
    raise CredentialUnavailableError(message=message)
azure.identity._exceptions.CredentialUnavailableError: EnvironmentCredential authentication unavailable. Environment variables ar

Successfully acquired token!


### Set Up Your Environment Variables

Set the following variables for use in this notebook. These variables connect to your Azure resources and model deployments.

Set these variables by creating an `.env` file in your project's root folder.

**Note:** You can find these values in your Azure AI Foundry project or Azure OpenAI resource.

For reference, here's an example of what your populated environment variables should look like:

```
# Azure OpenAI
AZURE_OPENAI_API_KEY="your-api-key-here"
AZURE_OPENAI_ENDPOINT="https://endpoint-name.cognitiveservices.azure.com/"
AZURE_OPENAI_DEPLOYMENT_NAME="gpt-4"
AZURE_OPENAI_API_VERSION="2024-12-01-preview"

# Azure AI Project
AZURE_PROJECT_ENDPOINT="https://your-aifoundry-endpoint-name.services.ai.azure.com/api/projects/yourproject-name"
```

In [5]:
# Import required packages for loading environment variables
from dotenv import load_dotenv
import os
from pathlib import Path

# Load environment variables from .env file in the root directory
notebook_path = Path().absolute()
root_dir = notebook_path.parent  # Go up two levels to reach root
env_path = root_dir / '.env'
load_dotenv(env_path)

print(f"📁 Loading .env from: {env_path}")

# Azure AI Project information
azure_ai_project = os.environ.get("PROJECT_CONNECTION_STRING")

# Azure OpenAI deployment information
azure_openai_deployment = os.environ.get("MODEL_DEPLOYMENT_NAME")  # e.g., "gpt-4"
azure_openai_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.environ.get("AZURE_OPENAI_API_KEY")  # e.g., "your-api-key"
azure_openai_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")  # Use default if not set

# Print loaded values (without sensitive data)
print(f"✅ Azure AI Project: {'✓ Loaded' if azure_ai_project else '✗ Missing'}")
print(f"✅ OpenAI Deployment: {azure_openai_deployment if azure_openai_deployment else '✗ Missing'}")
print(f"✅ OpenAI Endpoint: {'✓ Loaded' if azure_openai_endpoint else '✗ Missing'}")
print(f"✅ OpenAI API Key: {'✓ Loaded' if azure_openai_api_key else '✗ Missing'}")
print(f"✅ API Version: {azure_openai_api_version}")

📁 Loading .env from: /home/brian/working/ai-foundry-e2e-lab/.env
✅ Azure AI Project: ✓ Loaded
✅ OpenAI Deployment: gpt-4o
✅ OpenAI Endpoint: ✗ Missing
✅ OpenAI API Key: ✗ Missing
✅ API Version: 2024-12-01-preview


## Understanding AI Red Teaming Agent's capabilities

The Azure AI Evaluation SDK's `RedTeam` functionality evaluates AI systems against adversarial prompts across multiple dimensions:

1. **Risk Categories**: Different content risk categories your AI system might generate
   - Violence
   - HateUnfairness
   - Sexual
   - SelfHarm

2. **Attack Strategies**: Along with standard unmodified prompts which are sent by default as the `baseline`, you can specify different transformations of prompts to elicit undesired content.
You can also use `AttackStrategy.Compose()` to layer two strategies in one attack
   - AnsiAttack: Using ANSI escape codes in prompts
   - AsciiArt: Using ASCII art to disguise harmful content
   - AsciiSmuggler: Hiding harmful content within ASCII characters
   - Atbash: Using the Atbash cipher to encode harmful requests
   - Base64: Encoding harmful content in Base64 format
   - Binary: Converting text to binary to bypass filters
   - Caesar: Using the Caesar cipher for encoding
   - CharacterSpace: Manipulating character spacing to confuse filters
   - CharSwap: Swapping characters to bypass detection
   - Diacritic: Using diacritical marks to alter text appearance
   - Flip: Flipping text to bypass content filters
   - Leetspeak: Converting letters to numbers and symbols
   - Morse: Using Morse code to encode harmful requests
   - ROT13: Using ROT13 cipher for text transformation
   - SuffixAppend: Adding suffixes to confuse detection systems
   - StringJoin: Joining strings in unconventional ways
   - Tense: Changing the tense of harmful requests to past tense
   - UnicodeConfusable: Using similar-looking Unicode characters
   - UnicodeSubstitution: Substituting characters with Unicode alternatives
   - Url: Embedding harmful content within URLs
   - Jailbreak: Specially crafted prompts to bypass AI safeguards

3. **Complexity Levels**: Different difficultly levels of attacks
   - Baseline: Standard functionality tests
   - Easy: Simple attack patterns
   - Moderate: More sophisticated attacks
   - Difficult: Complex, layered attack strategies

The key metric for evaluating results is the **Attack Success Rate (ASR)**, which measures the percentage of attacks that successfully elicit harmful content from your AI system.

## Basic Example: Fixed Response Callback

Let's start with a basic example that demonstrates how to set up a Red Team evaluation using a simple callback that always returns a fixed, safe response regardless of the input prompt.

**Important Note**: Since this callback always responds with the same safe message, the Attack Success Rate (ASR) will be 0% for all categories. This example is merely to demonstrate the API structure and flow.

In [6]:
# Define a simple callback function that always returns a fixed response
def financial_advisor_callback(query: str) -> str:  # noqa: ARG001
    return "I'm a financial advisor assistant. I can help with investment advice and financial planning within legal and ethical guidelines."

In [7]:
# Create the `RedTeam` instance with minimal configurations
red_team = RedTeam(
    azure_ai_project=azure_ai_project,
    credential=credential,
    risk_categories=[RiskCategory.Violence, RiskCategory.HateUnfairness],
    num_objectives=1,
)

INFO:dotenv.main:python-dotenv could not find configuration file .env.
INFO:pyrit.common.initialization:Using persistent DuckDB database.
INFO:pyrit.memory.duckdb_memory:Engine created successfully for database: /home/brian/.local/share/dbdata/pyrit_duckdb_storage.db
INFO:pyrit.memory.central_memory:Central memory instance set to: DuckDBMemory
DEBUG:RedTeamLogger:RedTeam initialized successfully


NOTE: `num_objectives` specifies the number of attacks to perform per risk category per attack strategy. If the parameter `risk_categories` is not specified, `[RiskCategory.Violence, RiskCategory.HateUnfairness, RiskCategory.Sexual, RiskCategory.SelfHarm]` will be used by default.

Now let's run a simple automated scan using the `RedTeam` with the fixed response target. We'll test against two risk categories and one attack strategy for simplicity.

In [8]:
# Run the red team scan called "Basic-Callback-Scan" with limited scope for this basic example
# This will test 1 objective prompt for each of Violence and HateUnfairness categories with the Flip strategy
result = await red_team.scan(
    target=financial_advisor_callback,
    scan_name="Basic-Callback-Scan",
    attack_strategies=[AttackStrategy.Flip],
    output_path="red_team_output.json",
)

DEBUG:azure.identity._credentials.azure_cli:Executing subprocess with the following arguments ['/usr/bin/az', 'account', 'get-access-token', '--output', 'json', '--resource', 'https://ai.azure.com']


🚀 STARTING RED TEAM SCAN
📂 Output directory: ./.scan_Basic-Callback-Scan_20251002_125510
📊 Risk categories: ['violence', 'hate_unfairness']


INFO:azure.identity._internal.decorators:AzureCliCredential.get_token_info succeeded
DEBUG:azure.identity._internal.decorators:[Authenticated account] Client ID: 04b07795-8ddb-461a-bbee-02f9e1bf7b46. Tenant ID: 16b3c013-d300-468d-ac64-7eda0820b6d3. User Principal Name: unavailableUpn. Object ID (user): cd3cbaf1-9b7a-4b1a-9e19-a13a630b88fe
INFO:azure.identity._credentials.default:DefaultAzureCredential acquired a token from AzureCliCredential
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://werewolf-8910-foundry.services.ai.azure.com/api/projects/werewolf-8910-project/redTeams/runs:upload?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '38'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'f08deed4-9fb8-11f0-b5a2-00155d3236fd'
    'User-Agent': 'azure-ai-evaluation/1.11.1'
    'Authorization': 'REDACTED'
A body is sent with the request
DEBUG:urllib3.connectionpool:Starting

🔗 Track your red team scan in AI Foundry: https://ai.azure.com/resource/build/redteaming/f61f2926-00bd-4286-bd68-e089a9685649?wsid=/subscriptions/ccfc5dda-43af-4b5e-8cc2-1dda18f2382e/resourceGroups/werewolf-8910-ai_rg/providers/Microsoft.CognitiveServices/accounts/werewolf-8910-foundry/projects/werewolf-8910-project&tid=16b3c013-d300-468d-ac64-7eda0820b6d3
📋 Planning 4 total tasks


INFO:azure.identity._internal.decorators:AzureCliCredential.get_token_info succeeded
DEBUG:azure.identity._internal.decorators:[Authenticated account] Client ID: 04b07795-8ddb-461a-bbee-02f9e1bf7b46. Tenant ID: 16b3c013-d300-468d-ac64-7eda0820b6d3. User Principal Name: unavailableUpn. Object ID (user): cd3cbaf1-9b7a-4b1a-9e19-a13a630b88fe
INFO:azure.identity._credentials.default:DefaultAzureCredential acquired a token from AzureCliCredential
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://werewolf-8910-foundry.services.ai.azure.com/api/projects/werewolf-8910-project/redTeams/simulation/attackobjectives?api-version=REDACTED&riskTypes=REDACTED&riskCategory=REDACTED&lang=REDACTED'
Request method: 'GET'
Request headers:
    'x-ms-client-request-id': 'ae896787-7ea0-487b-b11a-2eb2fd3044fa'
    'Accept': 'application/json'
    'User-Agent': 'azure-ai-evaluation/1.11.1'
    'Authorization': 'REDACTED'
No body was attached to the request
DEBUG:urllib3.connectionpool:

...

Evaluation results saved to "/home/brian/working/ai-foundry-e2e-lab/ai-red-teaming-agent/.scan_Basic-Callback-Scan_20251002_125510/flip_violence_fc315f21-0546-4000-9197-0282f7acb735.json".
✅ Completed task 3/4 (75.0%) - flip/violence in 12.6s
   Est. remaining: 0.1 minutes


INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://werewolf-8910-foundry.services.ai.azure.com/api/projects/werewolf-8910-project/evaluations/operations/554b7120-2368-47dd-af72-7582f6abf96a?api-version=REDACTED'
Request method: 'GET'
Request headers:
    'Authorization': 'REDACTED'
    'User-Agent': 'azure-ai-evaluation/1.11.1 (type=redteam; subtype=RedTeam)'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'fb2fcd58-9fb8-11f0-b5a2-00155d3236fd'
No body was attached to the request
DEBUG:urllib3.connectionpool:https://werewolf-8910-foundry.services.ai.azure.com:443 "GET /api/projects/werewolf-8910-project/evaluations/operations/554b7120-2368-47dd-af72-7582f6abf96a?api-version=2025-05-15-preview HTTP/1.1" 200 None
INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'Content-Encoding': 'REDACTED'
    'Vary': 'REDACTED'

Evaluation results saved to "/home/brian/working/ai-foundry-e2e-lab/ai-red-teaming-agent/.scan_Basic-Callback-Scan_20251002_125510/flip_hate_unfairness_36c1d442-5e8e-44e0-b089-a859bfbaec5b.json".
✅ Completed task 4/4 (100.0%) - flip/hate_unfairness in 14.2s
   Est. remaining: 0.0 minutes


DEBUG:urllib3.connectionpool:https://werewolf-8910-foundry.services.ai.azure.com:443 "POST /api/projects/werewolf-8910-project/evaluationResults/3a5b410e-782e-4a4c-a1a9-85e15a551034/versions/1/startPendingUpload?api-version=2025-05-15-preview HTTP/1.1" 200 None
INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'Content-Encoding': 'REDACTED'
    'Vary': 'REDACTED'
    'Request-Context': 'REDACTED'
    'x-ms-response-type': 'REDACTED'
    'mise-correlation-id': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'azureml-served-by-cluster': 'REDACTED'
    'x-request-time': 'REDACTED'
    'apim-request-id': 'REDACTED'
    'x-ms-region': 'REDACTED'
    'Date': 'Thu, 02 Oct 2025 17:55:29 GMT'
DEBUG:azure.ai.evaluation._common.evaluation_onedp_client:Uploading /tmp/tmprjut0g0r to https://saoo7fp56pxjt4k.blob.co

Evaluation results saved to "/home/brian/working/ai-foundry-e2e-lab/ai-red-teaming-agent/red_team_output.json".

Evaluation results saved to "/home/brian/working/ai-foundry-e2e-lab/ai-red-teaming-agent/.scan_Basic-Callback-Scan_20251002_125510/final_results.json".

Overall ASR: 0.0%
Attack Success: 0/4 attacks were successful
------------------------------------------------------------------------------------------------------------------------------------
Risk Category        | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------------------------------
Violence             | 0.0%           | 0.0%                         | N/A                             | N/A                           
Hate-unfairness      | 0.0%           | 0.0%                         | N/A                             | N/A                        

## Intermediary Example: Using a Model Configuration as Target

Now let's create a more realistic example that uses an Azure OpenAI model for responding to the red teaming prompts. To test base or foundation models, you can update your target to take in a model configuration:

In [ ]:
# Define a model configuration to test
azure_oai_model_config = {
    "azure_endpoint": azure_openai_endpoint,
    "azure_deployment": azure_openai_deployment,
    "api_key": azure_openai_api_key,
}

Then, update your target to point to the model configurations and run the scan.

In [ ]:
# Run the red team scan called "Intermediary-Model-Target-Scan"
result = await red_team.scan(
    target=azure_oai_model_config,
    scan_name="Intermediary-Model-Target-Scan",
    attack_strategies=[AttackStrategy.Flip],
)

## Advanced Example: Using an Azure Open AI Model Endpoint in a Callback Function

Using the same Azure Open AI model configuration as above, we now wrap it in a callback function for more flexibility and control on the input and output handling. This will demonstrate how to evaluate an actual AI application. To test your own actual AI application, replace the inside of the callback function with a call to your application.

In [9]:
# Define a callback that uses Azure OpenAI API to generate responses
async def azure_openai_callback(
    messages: list,
    stream: Optional[bool] = False,  # noqa: ARG001
    session_state: Optional[str] = None,  # noqa: ARG001
    context: Optional[Dict[str, Any]] = None,  # noqa: ARG001
) -> dict[str, list[dict[str, str]]]:
    # Get token provider for Azure AD authentication
    token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

    # Initialize Azure OpenAI client
    client = AzureOpenAI(
        azure_endpoint=azure_openai_endpoint,
        api_version=azure_openai_api_version,
        azure_ad_token_provider=token_provider,
    )

    ## Extract the latest message from the conversation history
    messages_list = [{"role": message.role, "content": message.content} for message in messages]
    latest_message = messages_list[-1]["content"]

    try:
        # Call the model
        response = client.chat.completions.create(
            model=azure_openai_deployment,
            messages=[
                {"role": "user", "content": latest_message},
            ],
            # max_tokens=500, # If using an o1 base model, comment this line out
            max_completion_tokens=500,  # If using an o1 base model, uncomment this line
            # temperature=0.7, # If using an o1 base model, comment this line out (temperature param not supported for o1 base models)
        )

        # Format the response to follow the expected chat protocol format
        formatted_response = {"content": response.choices[0].message.content, "role": "assistant"}
    except Exception as e:
        print(f"Error calling Azure OpenAI: {e!s}")
        formatted_response = "I encountered an error and couldn't process your request."
    return {"messages": [formatted_response]}

In [ ]:
# Create the RedTeam instance with all of the risk categories with 5 attack objectives generated for each category
model_red_team = RedTeam(
    azure_ai_project=azure_ai_project,
    credential=credential,
    risk_categories=[RiskCategory.Violence, RiskCategory.HateUnfairness, RiskCategory.Sexual, RiskCategory.SelfHarm],
    num_objectives=5,
)

We will use this instance of `model_red_team` to test different attack strategies in the following section.

### Testing Different Attack Strategies

Now we'll run a more comprehensive evaluation using multiple attack strategies across risk categories. This will give us a better understanding of our model's vulnerabilities.

In [ ]:
# Run the red team scan with multiple attack strategies
advanced_result = await model_red_team.scan(
    target=azure_openai_callback,
    scan_name="Advanced-Callback-Scan",
    attack_strategies=[
        AttackStrategy.EASY,  # Group of easy complexity attacks
        AttackStrategy.MODERATE,  # Group of moderate complexity attacks
        AttackStrategy.CharacterSpace,  # Add character spaces
        AttackStrategy.ROT13,  # Use ROT13 encoding
        AttackStrategy.UnicodeConfusable,  # Use confusable Unicode characters
        AttackStrategy.CharSwap,  # Swap characters in prompts
        AttackStrategy.Morse,  # Encode prompts in Morse code
        AttackStrategy.Leetspeak,  # Use Leetspeak
        AttackStrategy.Url,  # Use URLs in prompts
        AttackStrategy.Binary,  # Encode prompts in binary
        AttackStrategy.Compose([AttackStrategy.Base64, AttackStrategy.ROT13]),  # Use two strategies in one attack
    ],
    output_path="Advanced-Callback-Scan.json",
)

The data and results used in this attack will be saved to the `output_path` specified. The URL printed out at the end of the scorecard will provide a link to where you results are uploaded and logged to your Azure AI Foundry project.

## Bring your own objectives: Using your own prompts as objectives for RedTeam

Below we demonstrate how to use your own prompts as objectives for a `RedTeam` scan. You can see the required format for prompts under `.\data\prompts.json`. Note that when bringing your own prompts, the supported `risk-type`s are `violence`, `sexual`, `hate_unfairness`, and `self_harm`. The number of prompts you specify will be the `num_objectives` used in the scan. 

In [ ]:
path_to_prompts = ".\data\prompts.json"

# Create the RedTeam specifying the custom attack seed prompts to use as objectives
custom_red_team = RedTeam(
    azure_ai_project=azure_ai_project,
    credential=credential,
    custom_attack_seed_prompts=path_to_prompts,  # Path to a file containing custom attack seed prompts
)

In [ ]:
custom_red_team_result = await custom_red_team.scan(
    target=azure_openai_callback,
    scan_name="Custom-Prompt-Scan",
    attack_strategies=[
        AttackStrategy.EASY,  # Group of easy complexity attacks
        AttackStrategy.MODERATE,  # Group of moderate complexity attacks
        AttackStrategy.DIFFICULT,  # Group of difficult complexity attacks
    ],
    output_path="Custom-Prompt-Scan.json",
)

## Conclusion

In this notebook, we've demonstrated how to use the Azure AI Evaluation SDK's `RedTeam` functionality to assess the safety and resilience of AI systems. We started with a basic fixed-response example and then moved to a more realistic model testing across multiple risk categories and attack strategies.

The automated AI red teaming scans provides valuable insights into:

1. **Overall Attack Success Rate (ASR)** - The percentage of attacks that successfully elicit harmful content
2. **Vulnerability by Risk Category** - Which types of harmful content your model is most vulnerable to
3. **Effectiveness of Attack Strategies** - Which attack techniques are most successful against your model
4. **Impact of Complexity** - How more sophisticated attacks affect your model's safety guardrails

By regularly red-teaming your AI applications, you can identify and address potential vulnerabilities before deploying your models to production environments.

### Next Steps

1. **Mitigation**: Use these results to strengthen your model's guardrails against identified attack vectors
2. **Continuous Testing**: Implement regular red team evaluations as part of your development lifecycle
3. **Custom Strategies**: Develop custom attack strategies for your specific use cases and domain
4. **Safety Layers**: Consider adding additional safety layers like Azure AI Content Safety to filter harmful requests and responses 